# 🇲🇦 Morocco Climate & Technology Dataset Downloader

**Colab-ready notebook** for downloading and preprocessing datasets for a Morocco climate exposure and technology access analysis.

| Group | Datasets |
|---|---|
| 🌡️ **Climate Exposure** | C1 CHIRPS · C2 MODIS LST · C3 MODIS NDVI · C4 Aridity Index |
| 📡 **Technology Access** | T1 Electricity · T2/T3 Telecom · T4 Roads (OSM) |
| 🗺️ **Boundaries** | GADM · HDX · Natural Earth |

> **How to use:** Run cells top-to-bottom. The pipeline is broken into independent cells so a single failure won't kill the whole run. Edit the **Configuration** cell to pick what to download.

> **Note on MODIS (C2/C3):** these require NASA Earthdata or Google Earth Engine authentication. The cells generate ready-to-run helper scripts rather than downloading directly.

## 1. Install dependencies

In [ ]:
# Colab already has: requests, pandas, numpy, tqdm, beautifulsoup4
# We need the geospatial stack. This takes ~60-90s on a fresh runtime.
import sys, subprocess

PKGS = ["geopandas", "rasterio", "rasterstats", "shapely", "fiona", "pyproj"]

def _pip_install(pkgs):
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", *pkgs],
        check=False,
    )

_pip_install(PKGS)
print("✅ Dependencies installed (or already present).")

## 2. Imports (with graceful degradation)

In [ ]:
import os
import sys
import zipfile
import gzip
import shutil
import logging
from pathlib import Path
from datetime import datetime
from typing import Optional

import requests
import pandas as pd
import numpy as np
from tqdm.auto import tqdm   # tqdm.auto renders correctly in Jupyter/Colab

# Optional geospatial imports — degrade gracefully if missing
try:
    import geopandas as gpd
    HAS_GEO = True
except ImportError:
    HAS_GEO = False
    print("⚠ geopandas not available — vector ops will be skipped.")

try:
    import rasterio
    from rasterio.mask import mask as rasterio_mask
    HAS_RASTERIO = True
except ImportError:
    HAS_RASTERIO = False
    print("⚠ rasterio not available — raster clipping will be skipped.")

try:
    from bs4 import BeautifulSoup
    HAS_BS4 = True
except ImportError:
    HAS_BS4 = False

print(f"HAS_GEO={HAS_GEO}  HAS_RASTERIO={HAS_RASTERIO}  HAS_BS4={HAS_BS4}")

## 3. (Optional) Mount Google Drive for persistent storage

By default, files are saved to `/content/morocco_data` on the Colab VM — **this is wiped when the runtime disconnects**. Uncomment the cell below to save everything to your Google Drive instead, so downloads survive across sessions.

In [ ]:
# Uncomment these 3 lines to use Google Drive for persistent storage:
# from google.colab import drive
# drive.mount('/content/drive')
# BASE_DIR_OVERRIDE = Path('/content/drive/MyDrive/morocco_data')

BASE_DIR_OVERRIDE = None  # leave as None to use local Colab storage

## 4. Configuration — edit these to control what runs

This replaces the original script's CLI arguments. Just change the variables and re-run.

In [ ]:
# ─── WHAT TO DOWNLOAD ────────────────────────────────────────────────
# Options for TARGETS:
#   "all"      → everything
#   "climate"  → C1, C2, C3, C4
#   "tech"     → T1, T2, T4
#   "admin"    → administrative boundaries
#   list of IDs → e.g. ["C1", "T4", "ADMIN"]
TARGETS = ["ADMIN", "T1", "T4"]   # ← safe defaults that actually download data

# ─── TEMPORAL RANGE for climate data ────────────────────────────────
YEAR_START = 2015
YEAR_END   = 2023

# ─── PATHS ──────────────────────────────────────────────────────────
BASE_DIR = BASE_DIR_OVERRIDE or Path("/content/morocco_data")
RAW_DIR        = BASE_DIR / "raw"
PROCESSED_DIR  = BASE_DIR / "processed"
BOUNDARIES_DIR = BASE_DIR / "boundaries"
LOG_FILE       = BASE_DIR / "download_log.txt"

# ─── MOROCCO BOUNDING BOX (approximate) ─────────────────────────────
MOROCCO_BBOX = {"west": -13.2, "south": 27.6, "east": -0.9, "north": 35.9}

# ─── BEHAVIOUR ──────────────────────────────────────────────────────
OVERWRITE = False  # set True to re-download files that already exist

# ─── LOGGING ────────────────────────────────────────────────────────
for h in list(logging.getLogger().handlers):
    logging.getLogger().removeHandler(h)
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    handlers=[logging.StreamHandler()],
)
logger = logging.getLogger("morocco")

# Make the dirs now so logging works
for d in [RAW_DIR, PROCESSED_DIR, BOUNDARIES_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print(f"📁 BASE_DIR = {BASE_DIR}")
print(f"🎯 TARGETS  = {TARGETS}")
print(f"📅 Years    = {YEAR_START}-{YEAR_END}")

## 5. Utility functions

In [ ]:
def download_file(url: str, dest: Path, chunk_size: int = 8192,
                  overwrite: bool = None) -> bool:
    """Download a file with a progress bar. Returns True on success."""
    if overwrite is None:
        overwrite = OVERWRITE
    if dest.exists() and not overwrite:
        logger.info(f"Already exists, skipping: {dest.name}")
        return True

    try:
        resp = requests.get(
            url, stream=True, timeout=60,
            headers={"User-Agent": "MoroccoDataDownloader/1.0"},
        )
        resp.raise_for_status()
        total = int(resp.headers.get("content-length", 0))

        with open(dest, "wb") as f, tqdm(
            total=total, unit="B", unit_scale=True,
            desc=dest.name[:40], leave=False
        ) as bar:
            for chunk in resp.iter_content(chunk_size=chunk_size):
                f.write(chunk)
                bar.update(len(chunk))

        size_mb = dest.stat().st_size / 1e6
        logger.info(f"Downloaded: {dest.name} ({size_mb:.1f} MB)")
        return True

    except requests.RequestException as e:
        logger.error(f"Failed to download {url}: {e}")
        if dest.exists():
            dest.unlink()
        return False


def extract_zip(zip_path: Path, dest_dir: Path):
    with zipfile.ZipFile(zip_path, "r") as zf:
        zf.extractall(dest_dir)
    logger.info(f"Extracted: {zip_path.name} → {dest_dir}")


def extract_gz(gz_path: Path, dest_path: Optional[Path] = None):
    if dest_path is None:
        dest_path = gz_path.with_suffix("")
    with gzip.open(gz_path, "rb") as f_in, open(dest_path, "wb") as f_out:
        shutil.copyfileobj(f_in, f_out)
    logger.info(f"Extracted: {gz_path.name} → {dest_path.name}")
    return dest_path


def clip_raster_to_morocco(raster_path: Path, output_path: Path,
                           boundary_path: Optional[Path] = None):
    """Clip a raster to Morocco's extent."""
    if not HAS_RASTERIO or not HAS_GEO:
        logger.warning("rasterio/geopandas not installed — skipping clip.")
        return False
    try:
        if boundary_path and boundary_path.exists():
            gdf = gpd.read_file(boundary_path)
            geometry = gdf.geometry.values
        else:
            from shapely.geometry import box
            geometry = [box(
                MOROCCO_BBOX["west"], MOROCCO_BBOX["south"],
                MOROCCO_BBOX["east"], MOROCCO_BBOX["north"],
            )]

        with rasterio.open(raster_path) as src:
            out_image, out_transform = rasterio_mask(
                src, geometry, crop=True, nodata=src.nodata or -9999
            )
            out_meta = src.meta.copy()
            out_meta.update({
                "driver": "GTiff",
                "height": out_image.shape[1],
                "width":  out_image.shape[2],
                "transform": out_transform,
                "compress": "lzw",
            })

        with rasterio.open(output_path, "w", **out_meta) as dst:
            dst.write(out_image)
        logger.info(f"Clipped raster → {output_path.name}")
        return True
    except Exception as e:
        logger.error(f"Clipping failed for {raster_path.name}: {e}")
        return False


def log_status(dataset_id: str, status: str, notes: str = ""):
    with open(LOG_FILE, "a") as f:
        ts = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
        f.write(f"{ts} | {dataset_id:6s} | {status:10s} | {notes}\n")

print("✅ Utilities ready.")

## 6. C1 — CHIRPS Precipitation (monthly)

Used to compute SPI-12 (Standardised Precipitation Index) for drought.

> ⚠️ **Heads up on disk usage:** a full 9-year download is roughly 1–2 GB depending on coverage. If on the free Colab tier, consider reducing `YEAR_START`/`YEAR_END` first.

In [ ]:
def download_c1_chirps(year_start: int = None, year_end: int = None):
    """Download monthly CHIRPS precipitation GeoTIFFs."""
    year_start = year_start or YEAR_START
    year_end   = year_end   or YEAR_END

    logger.info("═" * 60)
    logger.info("C1 — CHIRPS Precipitation (monthly)")
    logger.info("═" * 60)

    dest = RAW_DIR / "C1_chirps"
    dest.mkdir(exist_ok=True)

    base_url = "https://data.chc.ucsb.edu/products/CHIRPS-2.0/global_monthly/tifs/"
    downloaded = 0

    months = [(y, m) for y in range(year_start, year_end + 1) for m in range(1, 13)]
    for year, month in tqdm(months, desc="CHIRPS months"):
        fname = f"chirps-v2.0.{year}.{month:02d}.tif.gz"
        gz_path  = dest / fname
        tif_path = dest / fname.replace(".gz", "")

        if tif_path.exists():
            downloaded += 1
            continue

        if download_file(base_url + fname, gz_path):
            try:
                extract_gz(gz_path, tif_path)
                gz_path.unlink()
                downloaded += 1
            except Exception as e:
                logger.error(f"Extract failed for {fname}: {e}")

    log_status("C1", "DONE" if downloaded > 0 else "PARTIAL",
               f"{downloaded} monthly TIFFs for {year_start}-{year_end}")
    print(f"\n  ▶ CHIRPS: {downloaded} files in {dest}")
    print("  ▶ Next step: compute SPI-12 with `pip install climate-indices`")
    return downloaded

## 7. C2 — MODIS LST (Land Surface Temperature → heat stress)

In [ ]:
def download_c2_modis_lst():
    """MODIS MOD11A2 needs auth — this writes helper scripts (Earthdata + GEE)."""
    logger.info("═" * 60)
    logger.info("C2 — MODIS LST (MOD11A2 v061)")
    logger.info("═" * 60)

    dest = RAW_DIR / "C2_modis_lst"
    dest.mkdir(exist_ok=True)

    earthdata_script = dest / "download_modis_lst.sh"
    gee_script       = dest / "download_modis_lst_gee.py"

    earthdata_script.write_text(
        "#!/bin/bash\n"
        "# MODIS MOD11A2 v061 — NASA Earthdata bulk download\n"
        "# Prereq: account at https://urs.earthdata.nasa.gov + ~/.netrc credentials\n"
        "# Morocco tiles: h17v05 h17v06 h18v05 h18v06\n"
        "\n"
        'TILES="h17v05 h17v06 h18v05 h18v06"\n'
        "YEAR_START=2015\n"
        "YEAR_END=2023\n"
        'BASE_URL="https://e4ftl01.cr.usgs.gov/MOLT/MOD11A2.061"\n'
        "\n"
        "for year in $(seq $YEAR_START $YEAR_END); do\n"
        "    for doy in $(seq -w 1 8 365); do\n"
        '        DATE=$(date -d "${year}-01-01 +$((10#$doy - 1)) days" +%Y.%m.%d 2>/dev/null)\n'
        '        [ -z "$DATE" ] && continue\n'
        "        for tile in $TILES; do\n"
        "            wget --load-cookies ~/.urs_cookies --save-cookies ~/.urs_cookies \\\n"
        "                 --auth-no-challenge=on --keep-session-cookies \\\n"
        '                 -r -np -nd -A "MOD11A2*${tile}*.hdf" \\\n'
        '                 "${BASE_URL}/${DATE}/" -P ./hdf/ 2>/dev/null\n'
        "        done\n"
        "    done\n"
        "done\n"
    )

    gee_script.write_text(
        "#!/usr/bin/env python3\n"
        '"""Download MODIS LST for Morocco via Google Earth Engine.\n'
        "Prereq: pip install earthengine-api && earthengine authenticate\n"
        '"""\n'
        "import ee\n"
        "ee.Initialize()\n"
        "\n"
        'morocco = ee.FeatureCollection("FAO/GAUL/2015/level0") \\\n'
        '    .filter(ee.Filter.eq("ADM0_NAME", "Morocco"))\n'
        "\n"
        "for year in range(2015, 2024):\n"
        '    coll = (ee.ImageCollection("MODIS/061/MOD11A2")\n'
        '            .filterDate(f"{year}-01-01", f"{year}-12-31")\n'
        '            .select("LST_Day_1km"))\n'
        "    img = coll.mean().multiply(0.02).subtract(273.15).clip(morocco)\n"
        "    task = ee.batch.Export.image.toDrive(\n"
        '        image=img, description=f"Morocco_LST_{year}",\n'
        '        folder="Morocco_Climate_Data", region=morocco.geometry(),\n'
        '        scale=1000, crs="EPSG:4326", maxPixels=1e9,\n'
        "    )\n"
        "    task.start()\n"
        '    print(f"Submitted: LST {year}")\n'
    )

    try:
        os.chmod(earthdata_script, 0o755)
        os.chmod(gee_script, 0o755)
    except Exception:
        pass

    log_status("C2", "SCRIPTS", "Generated download scripts (need Earthdata/GEE auth)")
    print(f"\n  ▶ Helper scripts written to: {dest}")
    print("  ▶ Recommended: use Google Earth Engine (free for research).")

## 8. C3 — MODIS NDVI (vegetation)

In [ ]:
def download_c3_modis_ndvi():
    """MOD13A1 16-day NDVI at 500m. Also auth-gated — writes GEE script."""
    logger.info("═" * 60)
    logger.info("C3 — MODIS NDVI (MOD13A1 v061)")
    logger.info("═" * 60)

    dest = RAW_DIR / "C3_modis_ndvi"
    dest.mkdir(exist_ok=True)

    gee_script = dest / "download_modis_ndvi_gee.py"
    gee_script.write_text(
        "#!/usr/bin/env python3\n"
        '"""Download MODIS NDVI for Morocco via Google Earth Engine."""\n'
        "import ee\n"
        "ee.Initialize()\n"
        "\n"
        'morocco = ee.FeatureCollection("FAO/GAUL/2015/level0") \\\n'
        '    .filter(ee.Filter.eq("ADM0_NAME", "Morocco"))\n'
        "\n"
        "for year in range(2015, 2024):\n"
        '    coll = (ee.ImageCollection("MODIS/061/MOD13A1")\n'
        '            .filterDate(f"{year}-01-01", f"{year}-12-31")\n'
        '            .select("NDVI"))\n'
        "    img = coll.mean().multiply(0.0001).clip(morocco)\n"
        "    task = ee.batch.Export.image.toDrive(\n"
        '        image=img, description=f"Morocco_NDVI_{year}",\n'
        '        folder="Morocco_Climate_Data", region=morocco.geometry(),\n'
        '        scale=500, crs="EPSG:4326", maxPixels=1e9,\n'
        "    )\n"
        "    task.start()\n"
        '    print(f"Submitted: NDVI {year}")\n'
    )
    try:
        os.chmod(gee_script, 0o755)
    except Exception:
        pass

    log_status("C3", "SCRIPTS", "Generated GEE download script")
    print(f"\n  ▶ GEE script: {gee_script}")

## 9. C4 — Global Aridity Index

In [ ]:
def download_c4_aridity():
    """Global Aridity Index v3 from Figshare (~1km, 1970–2000 baseline)."""
    logger.info("═" * 60)
    logger.info("C4 — Global Aridity Index")
    logger.info("═" * 60)

    dest = RAW_DIR / "C4_aridity"
    dest.mkdir(exist_ok=True)

    files = {
        "Global_AI_v3.tif":  "https://figshare.com/ndownloader/files/34377245",
        "Global_ET0_v3.tif": "https://figshare.com/ndownloader/files/34377248",
    }
    for fname, url in files.items():
        print(f"  Attempting: {fname}")
        download_file(url, dest / fname)

    # Optional clip
    boundary = BOUNDARIES_DIR / "gadm41_MAR_shp" / "gadm41_MAR_0.shp"
    ai_raw = dest / "Global_AI_v3.tif"
    if ai_raw.exists():
        clip_raster_to_morocco(
            ai_raw,
            PROCESSED_DIR / "C4_aridity_morocco.tif",
            boundary if boundary.exists() else None,
        )

    log_status("C4", "DONE", "Aridity index downloaded")
    print("\n  ▶ AI interpretation: <0.03 hyper-arid · 0.03–0.20 arid ·"
          " 0.20–0.50 semi-arid · 0.50–0.65 dry sub-humid · >0.65 humid")

## 10. T1 — Electricity Access (World Bank)

In [ ]:
def download_t1_electricity():
    """Electricity access % from World Bank API."""
    logger.info("═" * 60)
    logger.info("T1 — Electricity Access (World Bank)")
    logger.info("═" * 60)

    dest = RAW_DIR / "T1_electricity"
    dest.mkdir(exist_ok=True)

    wb_url = (
        "https://api.worldbank.org/v2/country/MA/indicator/EG.ELC.ACCS.ZS"
        "?format=json&per_page=100&date=2000:2023"
    )

    try:
        resp = requests.get(wb_url, timeout=30)
        resp.raise_for_status()
        data = resp.json()

        if len(data) > 1 and data[1]:
            records = [{
                "year": item["date"],
                "value": item["value"],
                "indicator": item["indicator"]["value"],
                "country":   item["country"]["value"],
            } for item in data[1]]

            df = pd.DataFrame(records).sort_values("year").reset_index(drop=True)
            df.to_csv(dest / "T1_electricity_access_worldbank.csv", index=False)
            df.to_csv(PROCESSED_DIR / "T1_electricity_access.csv", index=False)

            print("\n  ▶ Morocco electricity access (most recent 10 years):")
            print(df[df["value"].notna()].tail(10).to_string(index=False))
        else:
            logger.warning("No data from World Bank API")
    except Exception as e:
        logger.error(f"World Bank API error: {e}")

    log_status("T1", "PARTIAL",
               "World Bank national data OK; HCP provincial = manual download")
    print(f"\n  ▶ For provincial data, see HCP/RGPH-2014 at https://www.hcp.ma/")

## 11. T2 & T3 — Mobile & Internet (World Bank proxy)

In [ ]:
def download_t2_t3_telecom():
    """Telecom indicators via World Bank API."""
    logger.info("═" * 60)
    logger.info("T2/T3 — Telecom indicators")
    logger.info("═" * 60)

    dest = RAW_DIR / "T2_T3_telecom"
    dest.mkdir(exist_ok=True)

    indicators = {
        "IT.CEL.SETS.P2": "Mobile cellular subscriptions (per 100 people)",
        "IT.NET.USER.ZS": "Individuals using the Internet (% of population)",
        "IT.NET.BBND.P2": "Fixed broadband subscriptions (per 100 people)",
    }

    all_records = []
    for code_id, name in indicators.items():
        wb_url = (
            f"https://api.worldbank.org/v2/country/MA/indicator/{code_id}"
            f"?format=json&per_page=100&date=2010:2023"
        )
        try:
            resp = requests.get(wb_url, timeout=30)
            resp.raise_for_status()
            data = resp.json()
            if len(data) > 1 and data[1]:
                for item in data[1]:
                    all_records.append({
                        "year": item["date"],
                        "indicator_code": code_id,
                        "indicator_name": name,
                        "value": item["value"],
                    })
        except Exception as e:
            logger.error(f"Failed for {code_id}: {e}")

    if all_records:
        df = pd.DataFrame(all_records).sort_values(
            ["indicator_code", "year"]
        ).reset_index(drop=True)
        df.to_csv(dest / "T2_T3_telecom_worldbank.csv", index=False)
        df.to_csv(PROCESSED_DIR / "T2_T3_telecom_indicators.csv", index=False)

        pivot = df.pivot_table(
            index="year", columns="indicator_name",
            values="value", aggfunc="first",
        )
        print("\n  ▶ Morocco telecom (last 8 years):")
        print(pivot.dropna(how="all").tail(8).to_string())

    log_status("T2", "PARTIAL", "World Bank proxy; ANRT = manual")
    log_status("T3", "PARTIAL", "World Bank proxy; ANRT = manual")

## 12. T4 — Roads (OpenStreetMap via Geofabrik)

In [ ]:
def download_t4_roads():
    """Morocco OSM extract from Geofabrik (Shapefile)."""
    logger.info("═" * 60)
    logger.info("T4 — Road Infrastructure (OSM)")
    logger.info("═" * 60)

    dest = RAW_DIR / "T4_roads"
    dest.mkdir(exist_ok=True)

    url = "https://download.geofabrik.de/africa/morocco-latest-free.shp.zip"
    zip_path = dest / "morocco-latest-free.shp.zip"

    if download_file(url, zip_path):
        extract_dir = dest / "morocco_osm_shp"
        extract_dir.mkdir(exist_ok=True)
        extract_zip(zip_path, extract_dir)

        roads_shp = extract_dir / "gis_osm_roads_free_1.shp"
        if roads_shp.exists() and HAS_GEO:
            logger.info("Processing roads layer...")
            gdf = gpd.read_file(roads_shp)
            print(f"\n  ▶ Top road classes:")
            print(gdf["fclass"].value_counts().head(15).to_string())

            major = [
                "motorway", "trunk", "primary", "secondary", "tertiary",
                "motorway_link", "trunk_link", "primary_link", "secondary_link",
            ]
            major_roads = gdf[gdf["fclass"].isin(major)]
            out = PROCESSED_DIR / "T4_morocco_major_roads.gpkg"
            major_roads.to_file(out, driver="GPKG")
            logger.info(f"Saved major roads: {out.name} ({len(major_roads)} features)")
        elif not HAS_GEO:
            logger.warning("geopandas missing — skipped roads filtering.")

    log_status("T4", "DONE", "OSM roads downloaded and filtered")

## 13. Administrative Boundaries (GADM + HDX + Natural Earth)

In [ ]:
def download_admin_boundaries():
    """GADM, HDX/OCHA, and Natural Earth admin boundaries for Morocco."""
    logger.info("═" * 60)
    logger.info("ADMIN — Administrative Boundaries")
    logger.info("═" * 60)

    # ── 1. GADM v4.1 ──
    print("\n  [1/3] GADM Morocco boundaries...")
    gadm_url = "https://geodata.ucdavis.edu/gadm/gadm4.1/shp/gadm41_MAR_shp.zip"
    gadm_zip = BOUNDARIES_DIR / "gadm41_MAR_shp.zip"
    if download_file(gadm_url, gadm_zip):
        gadm_dir = BOUNDARIES_DIR / "gadm41_MAR_shp"
        gadm_dir.mkdir(exist_ok=True)
        extract_zip(gadm_zip, gadm_dir)
        log_status("GADM", "DONE", "Levels 0-3")

    # ── 2. HDX / OCHA ──
    print("\n  [2/3] HDX Morocco boundaries...")
    hdx_api_url = (
        "https://data.humdata.org/api/3/action/package_show?id=cod-ab-mar"
    )
    try:
        resp = requests.get(hdx_api_url, timeout=30)
        resp.raise_for_status()
        data = resp.json()
        if data.get("success") and data["result"].get("resources"):
            for resource in data["result"]["resources"]:
                if resource["format"].upper() in ("SHP", "GEOJSON", "ZIP"):
                    rname = resource["name"]
                    download_file(resource["url"], BOUNDARIES_DIR / f"hdx_{rname}")
            log_status("HDX", "DONE", "OCHA admin boundaries")
        else:
            logger.warning("Could not parse HDX API response")
    except Exception as e:
        logger.error(f"HDX download error: {e}")
        log_status("HDX", "FAILED", str(e))

    # ── 3. Natural Earth ──
    print("\n  [3/3] Natural Earth boundaries...")
    ne_url = ("https://naciscdn.org/naturalearth/10m/cultural/"
              "ne_10m_admin_0_countries.zip")
    ne_zip = BOUNDARIES_DIR / "ne_10m_admin_0_countries.zip"
    if download_file(ne_url, ne_zip):
        ne_dir = BOUNDARIES_DIR / "natural_earth"
        ne_dir.mkdir(exist_ok=True)
        extract_zip(ne_zip, ne_dir)
        log_status("NE", "DONE", "Natural Earth 10m")

## 14. 🚀 Run the pipeline

This cell replaces the original script's `main()` / argparse logic. It honours the `TARGETS` variable you set in cell 4, and **wraps each dataset in its own try/except** so one failure can't kill the rest of the run.

In [ ]:
DATASET_FUNCS = {
    "C1":    download_c1_chirps,
    "C2":    download_c2_modis_lst,
    "C3":    download_c3_modis_ndvi,
    "C4":    download_c4_aridity,
    "T1":    download_t1_electricity,
    "T2":    download_t2_t3_telecom,
    "T3":    download_t2_t3_telecom,   # same function covers both
    "T4":    download_t4_roads,
    "ADMIN": download_admin_boundaries,
}

GROUP_MAP = {
    "all":     ["C1", "C2", "C3", "C4", "T1", "T2", "T4", "ADMIN"],
    "climate": ["C1", "C2", "C3", "C4"],
    "tech":    ["T1", "T2", "T4"],
    "admin":   ["ADMIN"],
}

def resolve_targets(targets):
    """Accept a string ('all'/'climate'/'tech'/'admin') or a list of IDs."""
    if isinstance(targets, str):
        if targets not in GROUP_MAP:
            raise ValueError(
                f"Unknown group: {targets!r}. "
                f"Use one of {list(GROUP_MAP)} or a list of IDs."
            )
        return list(GROUP_MAP[targets])
    if isinstance(targets, (list, tuple, set)):
        bad = [t for t in targets if t not in DATASET_FUNCS]
        if bad:
            raise ValueError(
                f"Unknown dataset IDs: {bad}. Valid: {list(DATASET_FUNCS)}"
            )
        return list(targets)
    raise TypeError(f"TARGETS must be str or list, got {type(targets).__name__}")


def run_pipeline(targets):
    resolved = resolve_targets(targets)
    logger.info(f"▶ Running targets: {resolved}")
    logger.info(f"▶ Years: {YEAR_START}-{YEAR_END}")

    results = {}
    seen_funcs = set()
    for dataset_id in resolved:
        func = DATASET_FUNCS[dataset_id]
        if func in seen_funcs:
            results[dataset_id] = "skipped (covered by sibling)"
            continue
        seen_funcs.add(func)

        try:
            if dataset_id == "C1":
                func(YEAR_START, YEAR_END)
            else:
                func()
            results[dataset_id] = "✅ done"
        except Exception as e:
            logger.error(f"Error processing {dataset_id}: {e}")
            log_status(dataset_id, "ERROR", str(e))
            results[dataset_id] = f"❌ {type(e).__name__}: {e}"

    print("\n" + "═" * 60)
    print("PIPELINE RESULTS")
    print("═" * 60)
    for k, v in results.items():
        print(f"  {k:6s}  {v}")
    return results

results = run_pipeline(TARGETS)

## 15. Final summary — what's on disk

In [ ]:
def run_preprocessing_summary():
    logger.info("═" * 60)
    logger.info("SUMMARY — Dataset Status")
    logger.info("═" * 60)

    checks = [
        ("C1",    "CHIRPS Precipitation", RAW_DIR / "C1_chirps"),
        ("C2",    "MODIS LST",            RAW_DIR / "C2_modis_lst"),
        ("C3",    "MODIS NDVI",           RAW_DIR / "C3_modis_ndvi"),
        ("C4",    "Aridity Index",        RAW_DIR / "C4_aridity"),
        ("T1",    "Electricity Access",   RAW_DIR / "T1_electricity"),
        ("T2/T3", "Telecom Indicators",   RAW_DIR / "T2_T3_telecom"),
        ("T4",    "Road Infrastructure",  RAW_DIR / "T4_roads"),
        ("GADM",  "Admin Boundaries",     BOUNDARIES_DIR / "gadm41_MAR_shp"),
    ]
    rows = []
    for vid, name, path in checks:
        if path.exists():
            n = sum(1 for p in path.rglob("*") if p.is_file())
            status = "✅ Ready" if n > 0 else "📁 Dir only"
        else:
            n, status = 0, "❌ Not downloaded"
        rows.append({"ID": vid, "Dataset": name, "Files": n, "Status": status})

    df = pd.DataFrame(rows)
    print("\n" + df.to_string(index=False))

    processed = list(PROCESSED_DIR.glob("*"))
    if processed:
        print(f"\n  Processed outputs ({len(processed)} files):")
        for f in processed:
            print(f"    {f.name} ({f.stat().st_size / 1e6:.1f} MB)")

    if LOG_FILE.exists():
        print(f"\n  Full log: {LOG_FILE}")

run_preprocessing_summary()

## 16. Next steps (post-processing hints)

### C1 → SPI-12
```python
!pip install climate-indices
from climate_indices import compute, indices
spi = indices.spi(
    precip_array, scale=12,
    distribution=indices.Distribution.gamma,
    data_start_year=YEAR_START,
    calibration_year_initial=YEAR_START,
    calibration_year_final=YEAR_END,
    periodicity=compute.Periodicity.monthly,
)
```

### T4 → road density per province
```python
import geopandas as gpd
roads = gpd.read_file(PROCESSED_DIR / "T4_morocco_major_roads.gpkg")
admin = gpd.read_file(BOUNDARIES_DIR / "gadm41_MAR_shp" / "gadm41_MAR_2.shp")
roads_m = roads.to_crs(epsg=32629)   # UTM 29N (metric)
admin_m = admin.to_crs(epsg=32629)
joined  = gpd.sjoin(roads_m, admin_m)
road_km = joined.groupby("NAME_2").geometry.apply(lambda g: g.length.sum() / 1000)
admin_m["area_km2"]     = admin_m.area / 1e6
admin_m["road_density"] = road_km / admin_m["area_km2"]
```

### Manual sources still needed
- **HCP provincial electrification** → https://www.hcp.ma/ (search "électricité" / "équipement des ménages")
- **ANRT regional telecom** → https://www.anrt.ma/publications/rapport-annuel
- **MODIS C2/C3** → run the generated GEE scripts after `earthengine authenticate`